In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

print("Librerías importadas correctamente ✅")

Librerías importadas correctamente ✅


## Experimentos — ¿Se puede mejorar el modelo?

El modelo base nos dio RMSE $70,051. Probamos tres caminos para mejorar la precisión
sin cambiar el tipo de modelo.

In [2]:
# Experimento 1: Feature engineering (ratios entre variables)
# Cargo el dataset desde cero para no mezclar con el modelo base
df_exp1 = pd.read_csv('housing.csv')

# Imputo nulos
mediana_por_zona = df_exp1.groupby('ocean_proximity')['total_bedrooms'].median()
df_exp1['total_bedrooms'] = df_exp1['total_bedrooms'].fillna(df_exp1['ocean_proximity'].map(mediana_por_zona))

# Agrego los ratios
df_exp1['rooms_per_household'] = df_exp1['total_rooms'] / df_exp1['households']
df_exp1['bedrooms_per_room'] = df_exp1['total_bedrooms'] / df_exp1['households']
df_exp1['population_per_household'] = df_exp1['population'] / df_exp1['households']

# Convierto categorica
df_exp1 = pd.get_dummies(df_exp1, columns=['ocean_proximity'], drop_first=True)

X_exp1 = df_exp1.drop('median_house_value', axis=1)
y_exp1 = df_exp1['median_house_value']

X_train_e1, X_test_e1, y_train_e1, y_test_e1 = train_test_split(X_exp1, y_exp1, test_size=0.2, random_state=42)

scaler_e1 = StandardScaler()
X_train_e1 = scaler_e1.fit_transform(X_train_e1)
X_test_e1 = scaler_e1.transform(X_test_e1)

modelo_e1 = LinearRegression()
modelo_e1.fit(X_train_e1, y_train_e1)
y_pred_e1 = modelo_e1.predict(X_test_e1)

rmse_e1 = np.sqrt(mean_squared_error(y_test_e1, y_pred_e1))
print(f"Experimento 1 — Feature engineering: RMSE ${rmse_e1:,.0f}")
print(f"Modelo base:                          RMSE $70,051")

Experimento 1 — Feature engineering: RMSE $73,267
Modelo base:                          RMSE $70,051


In [3]:
# Experimento 2: Eliminar filas con precio $500,001 (censoring)
df_exp2 = pd.read_csv('housing.csv')

# Elimino las filas con el precio tope artificial
df_exp2 = df_exp2[df_exp2['median_house_value'] < 500001]

# Imputo nulos
mediana_por_zona = df_exp2.groupby('ocean_proximity')['total_bedrooms'].median()
df_exp2['total_bedrooms'] = df_exp2['total_bedrooms'].fillna(df_exp2['ocean_proximity'].map(mediana_por_zona))

# Convierto categorica
df_exp2 = pd.get_dummies(df_exp2, columns=['ocean_proximity'], drop_first=True)

X_exp2 = df_exp2.drop('median_house_value', axis=1)
y_exp2 = df_exp2['median_house_value']

X_train_e2, X_test_e2, y_train_e2, y_test_e2 = train_test_split(X_exp2, y_exp2, test_size=0.2, random_state=42)

scaler_e2 = StandardScaler()
X_train_e2 = scaler_e2.fit_transform(X_train_e2)
X_test_e2 = scaler_e2.transform(X_test_e2)

modelo_e2 = LinearRegression()
modelo_e2.fit(X_train_e2, y_train_e2)
y_pred_e2 = modelo_e2.predict(X_test_e2)

rmse_e2 = np.sqrt(mean_squared_error(y_test_e2, y_pred_e2))
print(f"Experimento 2 — Eliminar censoring: RMSE ${rmse_e2:,.0f}")
print(f"Modelo base:                         RMSE $70,051")

Experimento 2 — Eliminar censoring: RMSE $62,562
Modelo base:                         RMSE $70,051


In [4]:
# Experimento 3: Aplicar logaritmo al precio
df_exp3 = pd.read_csv('housing.csv')

# Imputo nulos
mediana_por_zona = df_exp3.groupby('ocean_proximity')['total_bedrooms'].median()
df_exp3['total_bedrooms'] = df_exp3['total_bedrooms'].fillna(df_exp3['ocean_proximity'].map(mediana_por_zona))

# Convierto categorica
df_exp3 = pd.get_dummies(df_exp3, columns=['ocean_proximity'], drop_first=True)

X_exp3 = df_exp3.drop('median_house_value', axis=1)

# Aplico logaritmo al precio — convierte la distribución asimétrica en más simétrica
y_exp3 = np.log(df_exp3['median_house_value'])

X_train_e3, X_test_e3, y_train_e3, y_test_e3 = train_test_split(X_exp3, y_exp3, test_size=0.2, random_state=42)

scaler_e3 = StandardScaler()
X_train_e3 = scaler_e3.fit_transform(X_train_e3)
X_test_e3 = scaler_e3.transform(X_test_e3)

modelo_e3 = LinearRegression()
modelo_e3.fit(X_train_e3, y_train_e3)
y_pred_e3 = modelo_e3.predict(X_test_e3)

# Revierto el logaritmo para comparar en dólares
y_pred_e3_real = np.exp(y_pred_e3)
y_test_e3_real = np.exp(y_test_e3)

rmse_e3 = np.sqrt(mean_squared_error(y_test_e3_real, y_pred_e3_real))
print(f"Experimento 3 — Logaritmo al precio: RMSE ${rmse_e3:,.0f}")
print(f"Modelo base:                          RMSE $70,051")

Experimento 3 — Logaritmo al precio: RMSE $89,030
Modelo base:                          RMSE $70,051


In [5]:
# Resumen de experimentos
r2_e1 = r2_score(y_test_e1, y_pred_e1)
r2_e2 = r2_score(y_test_e2, y_pred_e2)
r2_e3 = r2_score(y_test_e3_real, y_pred_e3_real)

print("=== Comparación de experimentos ===")
print(f"{'Modelo':<40} {'RMSE':>10} {'R²':>8}")
print("-" * 60)
print(f"{'Modelo base':<40} {'$70,051':>10} {'0.6255':>8}")
print(f"{'Exp 1 — Feature engineering':<40} ${rmse_e1:>9,.0f} {r2_e1:>8.4f}")
print(f"{'Exp 2 — Eliminar censoring':<40} ${rmse_e2:>9,.0f} {r2_e2:>8.4f}")
print(f"{'Exp 3 — Logaritmo al precio':<40} ${rmse_e3:>9,.0f} {r2_e3:>8.4f}")

=== Comparación de experimentos ===
Modelo                                         RMSE       R²
------------------------------------------------------------
Modelo base                                 $70,051   0.6255
Exp 1 — Feature engineering              $   73,267   0.5904
Exp 2 — Eliminar censoring               $   62,562   0.6075
Exp 3 — Logaritmo al precio              $   89,030   0.3951


## Conclusión de experimentos

De los tres experimentos, solo eliminar el censoring mejoró el modelo (RMSE $62,562 vs $70,051 base).
El R² del Exp 2 no es directamente comparable con el base porque se calculó sobre un dataset 
diferente — al sacar las filas de $500,001 cambia la variación total del dataset.

El logaritmo y el feature engineering no aportaron mejoras con regresión lineal.
El mejor resultado posible dentro de este modelo sigue siendo limitado — 
una recta no puede capturar la complejidad real del precio de una casa.